# Chạy Project DIP trên Google Colab / Kaggle

Notebook này giúp bạn thiết lập môi trường và chạy project [DIP](https://github.com/SGU-ML25/DIP) trên Google Colab hoặc Kaggle.

## Các bước thực hiện:
1. Clone repository từ GitHub.
2. Cài đặt các thư viện cần thiết.
3. Tải dataset từ Hugging Face.
4. Cập nhật cấu hình đường dẫn.
5. Chạy training và evaluation.

### 1. Clone Repository
Chạy cell dưới đây để lấy code mới nhất từ GitHub.

In [1]:
!git clone https://github.com/SGU-ML25/DIP.git
%cd DIP

Cloning into 'DIP'...
remote: Enumerating objects: 37, done.
remote: Total 37 (delta 0), reused 0 (delta 0), pack-reused 37 (from 1)
Receiving objects: 100% (37/37), 30.09 MiB | 20.40 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/DIP


### 2. Cài đặt thư viện
Cài đặt các thư viện cần thiết để chạy project.

In [2]:
!pip install opencv-python pillow matplotlib huggingface_hub tqdm scikit-learn

### 3. Tải Dataset từ Hugging Face
Dataset được lưu trữ tại: https://huggingface.co/datasets/solozy/DIP

In [4]:
import os
from huggingface_hub import snapshot_download

# Tạo thư mục data nếu chưa có
os.makedirs("data", exist_ok=True)

print("Đang tải dataset từ Hugging Face...")
snapshot_download(  
    repo_id="solozy/DIP", 
    repo_type="dataset", 
    local_dir="data",
    local_dir_use_symlinks=False
)
print("Tải dataset thành công!")

Đang tải dataset từ Hugging Face...


Fetching ... files: 0it [00:00, ?it/s]

HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/solozy/DIP/resolve/3cd215f6c96f5ffd3f7df49233eba06094652d47/wood/train/good/108.png
Rate limited. Waiting 149.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/solozy/DIP/resolve/3cd215f6c96f5ffd3f7df49233eba06094652d47/wood/train/good/109.png
Rate limited. Waiting 149.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/solozy/DIP/resolve/3cd215f6c96f5ffd3f7df49233eba06094652d47/wood/train/good/111.png
Rate limited. Waiting 149.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/solozy/DIP/resolve/3cd215f6c96f5ffd3f7df49233eba06094652d47/wood/train/good/110.png
Rate limited. Waiting 149.0s before retry [Retry 1/5].
HTTP Error 429 thrown while requesting HEAD https://huggingface.co/datasets/solozy/DIP/resolve/3cd215f6c96f5ffd3f7df49233eba06094652d47/wood

Tải dataset thành công!


### 4. Cấu hình Project
Tự động cập nhật `DATA_ROOT` trong `src/utils.py` để phù hợp với môi trường hiện tại.

In [5]:
import os
import re

utils_path = 'src/utils.py'
if os.path.exists(utils_path):
    with open(utils_path, 'r', encoding='utf-8') as f:
        content = f.read()
    
    # Thay đổi đường dẫn DATA_ROOT
    current_dir = os.getcwd()
    new_data_root = f'DATA_ROOT = "{current_dir}/data/"'
    content = re.sub(r'DATA_ROOT = ".*?"', new_data_root, content)
    
    with open(utils_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print(f'Đã cập nhật DATA_ROOT trong {utils_path} thành: {current_dir}/data/')
else:
    print(f'Không tìm thấy file {utils_path}')

Đã cập nhật DATA_ROOT trong src/utils.py thành: /content/DIP/data/


### 5. Chạy Training
Bạn có thể chọn category để train: `bottle`, `cable`, `capsule`, `carpet`, `grid`, `hazelnut`, `leather`, `metal_nut`, `pill`, `screw`, `tile`, `toothbrush`, `transistor`, `wood`, `zipper`.

In [6]:
# Thay đổi category tại đây
CATEGORY = "cable"

!export PYTHONPATH=$PYTHONPATH:$(pwd) && python src/train.py --category {CATEGORY}

Starting training for {CATEGORY} on cuda...
Traceback (most recent call last):
  File "/content/DIP/src/train.py", line 152, in <module>
    train(args.category)
  File "/content/DIP/src/train.py", line 19, in train
    dataloader = DataLoader(dataset, batch_size=Config.BATCH_SIZE, shuffle=True)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 394, in __init__
    sampler = RandomSampler(dataset, generator=generator)  # type: ignore[arg-type]
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/sampler.py", line 149, in __init__
    raise ValueError(
ValueError: num_samples should be a positive integer value, but got num_samples=0


### 6. Chạy Evaluation
Sau khi train xong, bạn có thể đánh giá model cho category tương ứng.

In [ ]:
# Thay đổi category tại đây
CATEGORY = "bottle"

!export PYTHONPATH=$PYTHONPATH:$(pwd) && python src/evaluate.py --category {CATEGORY}

### 7. Hiển thị kết quả 

In [ ]:
# Load và chạy thêm script test toàn bộ ảnh lỗi đã viết trước đó
from IPython.display import display, Image
print("\n--- Đang thực hiện kiểm thử trên TOÀN BỘ ảnh lỗi ---")

def show_top_results(category, num=20):
    res_dir = f"results"
    images = sorted([f for f in os.listdir(res_dir) if f.startswith(f"{category}_diag")])
    for img in images[:num]:
        print(f"📄 Kết quả chẩn đoán: {img}")
        display(Image(filename=os.path.join(res_dir, img)))
        print("-" * 80)

show_top_results(CATEGORY)

### 7. Lưu trữ kết quả (Tùy chọn trên Colab)
Nếu bạn muốn lưu model và kết quả vào Google Drive.

In [ ]:
from google.colab import drive
import shutil

# drive.mount('/content/drive')
# dest_path = '/content/drive/MyDrive/DIP_Results'
# os.makedirs(dest_path, exist_ok=True)
# shutil.copytree('checkpoints', os.path.join(dest_path, 'checkpoints'), dirs_exist_ok=True)
# shutil.copytree('results', os.path.join(dest_path, 'results'), dirs_exist_ok=True)
# print("Đã sao chép kết quả vào Google Drive.")